In [18]:
%reset -f

In [19]:
from __future__ import annotations
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.cross_decomposition import CCA
#from mvlearn.embed import CCA # for this one i need to put cca.xxx([Xp_train, Yp_train]) insetad of cca.xxx(Xp_train, Yp_train)
from scipy.stats import pearsonr 
from scipy import stats

###############################################################################
# Paths & constants
###############################################################################
PUPIL_ROOT = Path(r"C:\Users\cdd\Documents\Uni\Special_course\pupil_processed_clara")
EEG_ROOT   = Path(r"C:\Users\cdd\Documents\Uni\Special_course\code\Special_course\eeg_theta_hilbert")

FRONTAL_MIDLINE = ['AFz','AF3','AF4','Fz','F1','F2','F3','F4','FC3','FC1','FC2','FC4','Cz','C3','C1','C2','C4']

OUT_TRIALS  = Path("trial_level_cca_fixedlag.csv")
OUT_SUBJECT = Path("subject_best_lag.csv")

SUBJECTS = np.setdiff1d(np.arange(32, 64), [32, 37, 53, 61, 66, 78, 84, 90, 94, 96])
#SUBJECTS = np.setdiff1d(np.arange(32, 99), [32, 37, 53, 61, 66, 78, 84, 90, 94, 96])
SHIFTS = np.arange(-100, 101)      # ±1 s at 100 Hz → samples
WIN_OFFSET1 = 200                  # discard first 200 ms
WIN_OFFSET2 = 110                  # discard last 110 ms

###############################################################################
# Helper functions
###############################################################################

def normalise_eeg(x: np.ndarray) -> np.ndarray:
    """Centre each channel and scale so Σ x² = 1 over time×channels."""
    x = x - x.mean(axis=0, keepdims=True)
    scale = np.sqrt(np.mean(x**2))
    return x / scale


def cca_corr(eeg: np.ndarray, pupil: np.ndarray) -> float:
    """Canonical correlation (single component)."""
    cca = CCA(n_components=1, max_iter=1000, scale=False, tol=1e-6)
    cca.fit(eeg, pupil)
    u, v = cca.transform(eeg, pupil)
    return float(np.corrcoef(u[:, 0], v[:, 0])[0, 1])


In [20]:
def load_all_trials(
        sub: int,
        eeg_root: Path = EEG_ROOT,
        pupil_root: Path = PUPIL_ROOT,
        min_len: int = 40,
        max_len_diff: int = 30,
) -> list[tuple[np.ndarray, np.ndarray, dict]]:
    """
    Load *all* valid EEG-pupil trial pairs for one subject.

    Parameters
    ----------
    sub : int
        Numeric subject ID (e.g. 42).
    eeg_root, pupil_root : Path
        Roots of the pre-processed EEG and pupil folders.
    min_len : int
        Minimum number of samples a pupil trace must have to be accepted.
    max_len_diff : int
        Reject trial if |len(pupil)-len(eeg)| exceeds this.
    Returns
    -------
    trials : list of (eeg, pupil_z, meta)
        * eeg        - (T × n_channels) float64, already centred/scaled
        * pupil_z    - (T × 1) float64, per-trial z-scored
        * meta       - dict with subject/condition/load/epoch
    """
    trials = []
    sub_tag = f"sub-{sub:03d}"
    eeg_sub  = eeg_root   / sub_tag
    pupil_sub = pupil_root / sub_tag

    if not eeg_sub.exists():
        print(f"{sub_tag}: EEG folder missing - skipped")
        return trials

    # iterate condition (“control” / “memory”) and load (“05” / “09” / “13”)
    for cond_path in sorted(eeg_sub.iterdir()):
        if not cond_path.is_dir():
            continue
        for load_path in sorted(cond_path.iterdir()):
            if not load_path.is_dir():
                continue

            # matching pupil directory
            pupil_path = pupil_sub / cond_path.name / load_path.name
            if not pupil_path.exists():
                continue

            eeg_epochs   = sorted(load_path.glob("trial_*.csv"))
            pupil_epochs = sorted(pupil_path.glob("trial_*.csv"))
            common = {f.name for f in eeg_epochs} & {f.name for f in pupil_epochs}
            if not common:
                continue

            for fname in sorted(common):
                eeg_df = pd.read_csv(load_path / fname, comment="#", index_col=0)
                pupil_df = pd.read_csv(pupil_path / fname, comment="#",
                                       names=["time", "diameter_z"], index_col=0)

                eeg   = eeg_df.values.astype(float)
                pupil = pupil_df["diameter_z"].values.astype(float)

                # basic validity checks
                if len(pupil) < min_len or abs(len(pupil) - len(eeg)) > max_len_diff:
                    continue

                # normalise signals ----------------------------------------
                eeg_norm = normalise_eeg(eeg)           # your helper from before
                pupil_z  = ((pupil - pupil.mean()) / pupil.std(ddof=0))

                # same number of samples
                T = min(len(eeg_norm), len(pupil_z))
                eeg_norm = eeg_norm[0:T, :]  # (T × n_channels)
                pupil_z  = pupil_z[0:T].reshape(-1, 1)

                meta = {
                    "subject":   sub_tag,
                    "condition": cond_path.name,
                    "load":      int(load_path.name),
                    "epoch":     fname
                }
                trials.append((eeg_norm, pupil_z, meta))

    return trials

from typing import List, Tuple
import numpy as np

def split_trials_by_condition(
        trials: List[Tuple[np.ndarray, np.ndarray, dict]],
        memory_label: str = "memory",
        control_label: str = "control"
) -> Tuple[List[Tuple[np.ndarray, np.ndarray, dict]], List[Tuple[np.ndarray, np.ndarray, dict]]]:
    """
    Separate a mixed list of (eeg, pupil, meta) trial tuples into memory-condition and control-condition sub-lists.

    Parameters
    ----------
    trials : list of tuples
        Each tuple = (eeg_array, pupil_array, meta_dict).
        meta_dict must contain a key 'condition'.
    memory_label : str
        The value of meta['condition'] that marks a memory trial.
    control_label : str
        The value of meta['condition'] that marks a control trial.

    Returns
    -------
    memory_trials  : list[tuple]
    control_trials : list[tuple]
    """
    memory_trials  = []
    control_trials = []

    for eeg, pupil, meta in trials:
        cond = meta.get("condition", "").lower()
        if cond == memory_label:
            memory_trials.append((eeg, pupil, meta))
        elif cond == control_label:
            control_trials.append((eeg, pupil, meta))
        else: raise ValueError(f"Unknown condition label: {cond}")

    return memory_trials, control_trials

def split_trials_by_load(trials: List[Tuple[np.ndarray, np.ndarray, dict]]
                         ) -> Tuple[List[Tuple[np.ndarray, np.ndarray, dict]], List[Tuple[np.ndarray, np.ndarray, dict]], List[Tuple[np.ndarray, np.ndarray, dict]]]:
    """
    Separate a mixed list of (eeg, pupil, meta) trial tuples into 05, 09 and 13 load sub-lists.

    Parameters
    ----------
    trials : list of tuples
        Each tuple = (eeg_array, pupil_array, meta_dict).
        meta_dict must contain a key 'load'.

    Returns
    -------
    load_05_trials  : list[tuple]
    load_09_trials  : list[tuple]
    load_13_trials  : list[tuple]
    """
    load_05_trials = []
    load_09_trials = []
    load_13_trials = []

    for eeg, pupil, meta in trials:
        load = meta.get("load")
        if load == 5:
            load_05_trials.append((eeg, pupil, meta))
        elif load == 9:
            load_09_trials.append((eeg, pupil, meta))
        elif load == 13:
            load_13_trials.append((eeg, pupil, meta))
        else: raise ValueError(f"Unknown load label: {load}")

    return load_05_trials, load_09_trials, load_13_trials


In [21]:
import numpy as np
from typing import List, Tuple, Dict

def search_best_lag(
        train_trials: List[Tuple[np.ndarray, np.ndarray, dict]],
        shifts: np.ndarray = SHIFTS,
        return_curve: bool = False
) -> Tuple[float, int, Dict[int, float] | None]:
    """
    Search for the lag (sample shift) that maximises the mean canonical
    correlation between EEG and pupil traces in a training set.

    Parameters
    ----------
    train_trials : list of (eeg, pupil_z, meta)
        Each eeg  : 2-D array [time × channels or CCA-components]
        Each pupil: 1-D array [time]
    shifts : np.ndarray
        Array of integer lag shifts (positive = EEG is moved forward).
    return_curve : bool, default False
        If True, also return the full {shift: mean_r} dictionary.

    Returns
    -------
    best_corr : float
        Highest mean canonical correlation found.
    best_shift : int
        Shift (samples) that maximised the correlation.
    mean_r_per_shift : dict | None
        Only when `return_curve` is True.
    """
    win = slice(WIN_OFFSET1, -WIN_OFFSET2)         # common window
    mean_r_per_shift: Dict[int, float] = {}

    for s in shifts:                               # <-- loop over *shifts*, not SHIFTS
        rs = []
        for eeg, pupil_z, _ in train_trials:
            eeg_shifted = np.roll(eeg, s, axis=0)[win]
            r = cca_corr(eeg_shifted, pupil_z[win])
            rs.append(r)
        mean_r_per_shift[s] = float(np.mean(rs))   # cast to plain float for JSON-ability

    best_shift = max(mean_r_per_shift, key=mean_r_per_shift.get)
    best_corr  = mean_r_per_shift[best_shift]

    if return_curve:
        return best_corr, best_shift, mean_r_per_shift
    else:
        return best_corr, best_shift, None


In [22]:

from typing import List, Tuple, Optional

# trials  : list of (eeg, pupil_z, meta)   -- the tuples returned by load_all_trials
# shift   : integer sample shift (best_shift)
# win     : slice or None                  -- cropping window (set to None if the
#                                            trials are already pre-trimmed)
def concat_trials(
        trials: List[Tuple[np.ndarray, np.ndarray, dict]],
        shift: int = 0,
        win: Optional[slice] = None
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Concatenate a list of trials into one long matrix pair ready for CCA.

    Returns
    -------
    X  : ndarray, shape (Σ Tᵢ, n_channels)
    Y  : ndarray, shape (Σ Tᵢ, 1)
    """
    X_blocks, Y_blocks = [], []

    for eeg, pupil_z, _ in trials:
        # 1. roll *inside* the trial so samples never cross trial boundaries
        eeg_shift = np.roll(eeg, shift, axis=0)

        # 2. optional windowing (do it once here if you did **not** crop in loader)
        if win is not None:
            eeg_shift = eeg_shift[win]
            pupil_seg = pupil_z[win]
        else:
            pupil_seg = pupil_z          # already trimmed earlier

        # 3. stack
        X_blocks.append(eeg_shift)
        Y_blocks.append(pupil_seg)

    X = np.vstack(X_blocks)
    Y = np.vstack(Y_blocks)
    return X, Y

def iterate_trials(trials, shift, win):
    """Yield (eeg_shifted, pupil_z_windowed, meta) one by one."""
    for eeg, pupil_z, meta in trials:
        eeg_s = np.roll(eeg, shift, axis=0)[win]
        yield eeg_s, pupil_z[win], meta.copy()

import pandas as pd
import json
from pathlib import Path

def save_cca_weights(cca, subject_tag, lag_ms, eeg_ch_names, condition, out_dir=Path("weights")):
    """
    Dump EEG & pupil canonical weights to CSV/JSON for one subject.

    Parameters
    ----------
    cca            : fitted sklearn.cross_decomposition.CCA
    subject_tag    : "sub-042"
    lag_ms         : e.g. -90
    eeg_ch_names   : list[str] same order as columns in your trial matrices
    out_dir        : destination folder (created if missing)
    """
    out_dir.mkdir(parents=True, exist_ok=True)

    # ------- Pupil weight -> JSON ---------------------------------------
    w_pupil_og = float(cca.y_weights_[0, 0])   # scalar in 1-dim pupil case
    w_pupil = 1.0
    with open(out_dir / f"{subject_tag}_pupil_weight_{condition}_{lag_ms:+d}ms.json", "w") as fh:
        json.dump({"weight": w_pupil}, fh, indent=2)
    
    # ------- EEG weights -> tidy CSV ------------------------------------
    w_eeg = cca.x_weights_[:, 0] / w_pupil_og
    w_eeg = pd.Series(w_eeg, index=eeg_ch_names, name="weight")
    w_eeg.index.name = "channel"
    w_eeg.to_csv(out_dir / f"{subject_tag}_eeg_weights_{condition}_{lag_ms:+d}ms.csv")

    print(f"saved weights for {subject_tag} (lag {lag_ms:+d} ms)")



In [23]:
best_shift_by_sub = {33: -5, 34: 32, 35: -8, 36: -59, 38: -1, 39: 2, 40: 100, 41: -10, 42: -80, 43: 70, 44: -10, 45: 42, 46: -68, 47: 100, 48: -63, 49: 46, 50: 91, 51: 100, 52: 81, 54: 63, 55: -38, 56: 23, 57: 36, 58: 46, 59: -7, 60: 100, 62: -9, 63: 0, 64: 53, 65: -6, 67: 2, 68: 96, 69: 28, 70: 16, 71: 74, 72: 0, 73: 38, 74: -100, 75: 32, 76: 100, 77: 76, 79: 80, 80: -1, 81: 64, 82: -44, 83: -52, 85: 37, 86: 100, 87: 24, 88: 100, 89: -100, 91: -5, 92: 41, 93: 100, 95: -7, 97: -56, 98: 69}
print(best_shift_by_sub)

{33: -5, 34: 32, 35: -8, 36: -59, 38: -1, 39: 2, 40: 100, 41: -10, 42: -80, 43: 70, 44: -10, 45: 42, 46: -68, 47: 100, 48: -63, 49: 46, 50: 91, 51: 100, 52: 81, 54: 63, 55: -38, 56: 23, 57: 36, 58: 46, 59: -7, 60: 100, 62: -9, 63: 0, 64: 53, 65: -6, 67: 2, 68: 96, 69: 28, 70: 16, 71: 74, 72: 0, 73: 38, 74: -100, 75: 32, 76: 100, 77: 76, 79: 80, 80: -1, 81: 64, 82: -44, 83: -52, 85: 37, 86: 100, 87: 24, 88: 100, 89: -100, 91: -5, 92: 41, 93: 100, 95: -7, 97: -56, 98: 69}


### training and testing on all

In [ ]:
import random
from math import atanh, sqrt
from scipy.stats import norm, ttest_rel, wilcoxon

import matplotlib.pyplot as plt

all_rows = []  # collect all trial-level results here
# best_shift_by_sub = {}  # best shift per subject
rows_trials = []
rows_subject = []  
bad_subjects_mem = []
bad_subjects_ctrl = []

for subj in SUBJECTS:
    if subj == 57:
        continue
    print(f"Processing subject {subj:02d}...")
    trials = load_all_trials(subj)                         # list of (eeg, pupil)
    trials_memory, trials_control = split_trials_by_condition(trials)

    random.shuffle(trials_memory)
    random.shuffle(trials_control)

    trials_mem_05, trials_mem_09, trials_mem_13 = split_trials_by_load(trials_memory)
    trials_ctrl_05, trials_ctrl_09, trials_ctrl_13 = split_trials_by_load(trials_control)

    if len(trials_mem_05) < 2 or len(trials_mem_09) < 2 or len(trials_mem_13) < 2 or len(trials_ctrl_05) < 2 or len(trials_ctrl_09) < 2 or len(trials_ctrl_13) < 2:
        print(f"Subject {subj:02d} skipped - not enough trials in one of the load conditions.")
        continue

    mid_mem_05 = int(len(trials_mem_05) // 2)
    mid_mem_09 = int(len(trials_mem_09) // 2)
    mid_mem_13 = int(len(trials_mem_13) // 2)

    mid_ctrl_05 = int(len(trials_ctrl_05) // 2)
    mid_ctrl_09 = int(len(trials_ctrl_09) // 2)
    mid_ctrl_13 = int(len(trials_ctrl_13) // 2)

    train_memory = trials_mem_05[:mid_mem_05] + trials_mem_09[:mid_mem_09] + trials_mem_13[:mid_mem_13]
    test_memory  = trials_mem_05[mid_mem_05:] + trials_mem_09[mid_mem_09:] + trials_mem_13[mid_mem_13:]
    train_control = trials_ctrl_05[:mid_ctrl_05] + trials_ctrl_09[:mid_ctrl_09] + trials_ctrl_13[:mid_ctrl_13]
    test_control = trials_ctrl_05[mid_ctrl_05:] + trials_ctrl_09[mid_ctrl_09:] + trials_ctrl_13[mid_ctrl_13:]

    train_trials = train_memory + train_control
    random.shuffle(train_trials)

    best_shift = best_shift_by_sub[subj] if subj in best_shift_by_sub else print("MISTAKE")
    # ----- fit weights ONCE using all train trials at best_shift ---------
    win = slice(WIN_OFFSET1, -WIN_OFFSET2)

    best_shift = int(best_shift)  # convert to int if it was float
    X_train, Y_train = concat_trials(train_trials, shift=best_shift, win=win)
    X_mem_test, Y_mem_test = concat_trials(test_memory, shift=best_shift, win=win)
    X_ctrl_test, Y_ctrl_test = concat_trials(test_control, shift=best_shift, win=win)

    # --- fit CCA  ---
    cca = CCA(n_components=1, max_iter=1000, scale=False, tol=1e-6)
    cca.fit(X_train, Y_train)

    # --- transform MEMORY set  ---
    cvX_m, cvY_m = cca.transform(X_mem_test, Y_mem_test)
    r_mem, p_mem = pearsonr(cvX_m[:, 0], cvY_m[:, 0])

    # --- transform CONTROL set  ---
    cvX_c, cvY_c = cca.transform(X_ctrl_test, Y_ctrl_test)
    r_ctrl, p_ctrl = pearsonr(cvX_c[:, 0], cvY_c[:, 0])

    rows_subject.append({ 
        "subject": subj, 
        "condition": "memory", 
        "lag_ms": best_shift * 10, 
        "r": float(r_mem),
        "p-value": float(p_mem)
    })

    rows_subject.append({ 
        "subject": subj, 
        "condition": "control", 
        "lag_ms": best_shift * 10, 
        "r": float(r_ctrl),
        "p-value": float(p_ctrl)
    })

    n_mem  = cvX_m.shape[0]  # number of paired points used in r_mem
    n_ctrl = cvX_c.shape[0]  # number of paired points used in r_ctrl

    z_mem  = atanh(r_mem)
    z_ctrl = atanh(r_ctrl)

    # z-test for H1: r_mem > r_ctrl
    se = sqrt(1/(n_mem - 3) + 1/(n_ctrl - 3))
    z_stat = (z_mem - z_ctrl) / se
    p_one_sided = 1 - norm.cdf(z_stat)

    rows_subject.append({
        "subject": subj,
        "condition": "per-subject test",
        "lag_ms": best_shift * 10,
        "r_mem": float(r_mem),
        "r_ctrl": float(r_ctrl),
        "n_mem": int(n_mem),
        "n_ctrl": int(n_ctrl),
        "z_stat": float(z_stat),
        "p_one_sided": float(p_one_sided),
    })

    if p_ctrl > 0.05:
        bad_subjects_ctrl.append(subj)
    if p_mem > 0.05:
        bad_subjects_mem.append(subj)
    if p_mem > 0.05 and p_ctrl > 0.05:
        print(f"Subject {subj:02d} has both p_mem and p_ctrl > 0.05")

    save_cca_weights(cca, subject_tag=f"sub-{subj:03d}", lag_ms=best_shift*10, condition = "mem", eeg_ch_names=FRONTAL_MIDLINE, out_dir=Path("weights_HoldStratLoadHILBERT"))

##############################################################################
# ---- save -----------------------------------------------------------------
##############################################################################

##############################################################################
# 4)  after the loop - save per‑subject correlations
##############################################################################
pd.DataFrame(rows_subject).to_csv("subject_level_cca_fixedlagAllHilbert.csv", index=False)
print("Finished - saved subject-level correlations as subject_level_cca_fixedlagAllHilbert.csv.")


Processing subject 33...
Subject 33 has both r_mem and r_ctrl < 0.05
saved weights for sub-033 (lag -50 ms)
Processing subject 34...
Subject 34 has both r_mem and r_ctrl < 0.05
saved weights for sub-034 (lag +320 ms)
Processing subject 35...
Subject 35 skipped - not enough trials in one of the load conditions.
Processing subject 36...
Subject 36 has both r_mem and r_ctrl < 0.05
saved weights for sub-036 (lag -590 ms)
Processing subject 38...
saved weights for sub-038 (lag -10 ms)
Processing subject 39...
Subject 39 has both r_mem and r_ctrl < 0.05
saved weights for sub-039 (lag +20 ms)
Processing subject 40...
saved weights for sub-040 (lag +1000 ms)
Processing subject 41...
Subject 41 has both r_mem and r_ctrl < 0.05
saved weights for sub-041 (lag -100 ms)
Processing subject 42...
Subject 42 has both r_mem and r_ctrl < 0.05
saved weights for sub-042 (lag -800 ms)
Processing subject 43...
Subject 43 has both r_mem and r_ctrl < 0.05
saved weights for sub-043 (lag +700 ms)
Processing sub

In [25]:
df = pd.DataFrame(rows_subject)

# keep one row per subject per condition
wide = (df.query("condition in ['memory','control']")
          .pivot(index="subject", columns="condition", values="r"))

# subjects with non-significant per-subject test (p_one_sided > 0.05)
bad_subjects = df.loc[
    (df["condition"] == "per-subject test") & (df["p_one_sided"] > 0.05),
    "subject"
].unique()

print("There are", len (SUBJECTS) - len(bad_subjects), "subjects with significant per-subject test (p_one_sided < 0.05)")

# keep everyone else and make the wide table
wide2 = (df[~df["subject"].isin(bad_subjects)]
          .query("condition in ['memory','control']")
          .pivot(index="subject", columns="condition", values="r"))

# Fisher z
z_mem  = np.arctanh(wide["memory"])
z_ctrl = np.arctanh(wide["control"])
diff   = z_mem - z_ctrl

# one-sided paired t-test: H1 mean(diff) > 0
t2, p_two = ttest_rel(z_mem, z_ctrl, nan_policy='omit')
# convert to one-sided
p_one = p_two / 2 if diff.mean() > 0 else 1 - p_two/2

# (optional) distribution-free paired test on r (or z):
w_stat, p_wilcox_two = wilcoxon(z_mem, z_ctrl, alternative="greater", zero_method="wilcox", correction=False)

print({
    "mean_diff_z": float(np.nanmean(diff)),
    "t_stat": float(t2),
    "p_one_sided_t": float(p_one),
    "wilcoxon_W": float(w_stat),
    "p_one_sided_wilcoxon": float(p_wilcox_two),
    "n_subjects": int(diff.dropna().shape[0]),
})


There are 13 subjects with significant per-subject test (p_one_sided < 0.05)
{'mean_diff_z': 0.008359647553739873, 't_stat': 0.8347057313613502, 'p_one_sided_t': 0.20605774487723588, 'wilcoxon_W': 178.0, 'p_one_sided_wilcoxon': 0.34575942158699036, 'n_subjects': 25}


In [27]:
df_subj = pd.read_csv("subject_level_cca_fixedlagAllHilbert.csv")

# Calculate and print mean correlations for each condition
mean_memory = df_subj[df_subj['condition'] == 'memory']['r'].mean()
mean_control = df_subj[df_subj['condition'] == 'control']['r'].mean()
print(f"Mean correlation for memory condition: {mean_memory:.4f}")
print(f"Mean correlation for control condition: {mean_control:.4f}")
print()

# Pivot so we have columns for memory and control per subject
df_wide = df_subj.pivot(index="subject", columns="condition", values="r")
print(df_wide.head(20))

# --- 95% CI for the paired mean difference (mem − ctl) ---
paired = df_wide.dropna(subset=["memory", "control"])
diff   = paired["memory"] - paired["control"]

n      = diff.size
dfree  = n - 1
mean_d = diff.mean()
se_d   = diff.std(ddof=1) / np.sqrt(n)

alpha  = 0.05
tcrit  = stats.t.ppf(1 - alpha/2, dfree)   # two-sided 95%
ci_low, ci_high = mean_d - tcrit*se_d, mean_d + tcrit*se_d

# (optional) re-run t-test on the same paired subset
t2, p2 = stats.ttest_rel(paired["memory"], paired["control"])
p_one  = p2/2 if t2 > 0 else 1 - p2/2

print(f"Paired t-test: t={t2:.3f}, p_two={p2:.4f}, "
      f"p_one(mem>ctl)={p_one:.4f}, df={dfree}, "
      f"meanΔ={mean_d:.3f}, 95% CI=[{ci_low:.3f}, {ci_high:.3f}]")


Mean correlation for memory condition: 0.0301
Mean correlation for control condition: 0.0217

condition   control    memory  per-subject test
subject                                        
33        -0.017906  0.001909               NaN
34        -0.034236  0.030826               NaN
36        -0.037999  0.039436               NaN
38         0.094335  0.025163               NaN
39         0.033326  0.008632               NaN
40         0.197792  0.125824               NaN
41         0.044459  0.028599               NaN
42        -0.024576  0.032435               NaN
43         0.041281  0.020992               NaN
44         0.046199  0.016868               NaN
45         0.028109  0.015323               NaN
47         0.068798  0.068399               NaN
48         0.020935 -0.020506               NaN
49         0.042592  0.049672               NaN
50         0.065812  0.126494               NaN
51        -0.069874  0.061426               NaN
52         0.000605  0.061556             

### Now with k folds (k = 3) and stratifying condition and load

In [ ]:
import numpy as np
rng = np.random.RandomState(42)  # reproducible shuffles

def _trial_tag(x):
    """Best-effort short tag for printing a trial object without huge dumps."""
    for attr in ("trial_id", "id", "name"):
        if hasattr(x, attr):
            return f"{attr}={getattr(x, attr)}"
    # fallback: try tuple-like or object id
    try:
        return f"tuple0={x[0]!r}"
    except Exception:
        return f"obj@{hex(id(x))}"

def _chunk_k(lst, k, rng):
    """Shuffle lst and split into k chunks (as even as possible), purely in Python."""
    idx = list(range(len(lst)))
    rng.shuffle(idx)
    shuffled = [lst[i] for i in idx]
    n = len(shuffled)
    base, rem = divmod(n, k)
    chunks, start = [], 0
    for f in range(k):
        size = base + (1 if f < rem else 0)
        chunks.append(shuffled[start:start+size])
        start += size
    return chunks

def stratified_kfold_trials(trials_memory, trials_control, k=3, verbose=True, show_examples=2):
    """
    Stratified K-fold by condition × load. Returns a list of fold dicts:
      {"train_memory", "test_memory", "train_control", "test_control"}.
    Debug prints show bucket counts, chunk sizes, per-fold sizes, and integrity checks.
    """
    # --- split memory/control by load ---
    tm05, tm09, tm13 = split_trials_by_load(trials_memory)
    tc05, tc09, tc13 = split_trials_by_load(trials_control)

    buckets = {
        ("memory", 5): list(tm05), ("memory", 9): list(tm09), ("memory", 13): list(tm13),
        ("control", 5): list(tc05), ("control", 9): list(tc09), ("control", 13): list(tc13),
    }

    order = [("memory",5),("memory",9),("memory",13),("control",5),("control",9),("control",13)]

    if verbose:
        print("\n[Bucket counts]")
        for key in order:
            print(f"  {key}: {len(buckets[key])}")

    # need at least k trials per bucket so each fold gets ≥1 in its test
    too_small = [key for key in buckets if len(buckets[key]) < k]
    if too_small:
        if verbose:
            print(f"[WARN] Not enough trials for k={k} in buckets: {too_small}. Returning [].")
        return []

    # --- split each bucket into k chunks ---
    parts = {key: _chunk_k(buckets[key], k, rng) for key in buckets}

    if verbose:
        print("\n[Chunk sizes per bucket]  (each list has k entries = fold-wise test sizes)")
        for key in order:
            sizes = [len(ch) for ch in parts[key]]
            print(f"  {key}: sizes={sizes}  total={sum(sizes)}")
            # optional: show example trial tags from first chunk
            if show_examples and len(parts[key][0]) > 0:
                ex = ", ".join(_trial_tag(t) for t in parts[key][0][:show_examples])
                print(f"      examples from fold-1 test chunk: [{ex}]")

    # --- build folds ---
    folds = []
    for f in range(k):
        test_memory  = list(parts[("memory", 5)][f]) + list(parts[("memory", 9)][f]) + list(parts[("memory", 13)][f])
        test_control = list(parts[("control", 5)][f]) + list(parts[("control", 9)][f]) + list(parts[("control", 13)][f])

        train_memory  = [x for i in range(k) if i != f for x in parts[("memory", 5)][i]] \
                      + [x for i in range(k) if i != f for x in parts[("memory", 9)][i]] \
                      + [x for i in range(k) if i != f for x in parts[("memory", 13)][i]]
        train_control = [x for i in range(k) if i != f for x in parts[("control", 5)][i]] \
                      + [x for i in range(k) if i != f for x in parts[("control", 9)][i]] \
                      + [x for i in range(k) if i != f for x in parts[("control", 13)][i]]

        folds.append({
            "train_memory":  train_memory,
            "test_memory":   test_memory,
            "train_control": train_control,
            "test_control":  test_control,
        })

    # --- per-fold debug summary ---
    if verbose:
        print("\n[Fold summaries]")
        for f, fold in enumerate(folds, 1):
            # count by load inside each set (memory/control)
            def _count_by_load(trials):
                a5, a9, a13 = split_trials_by_load(trials)
                return len(a5), len(a9), len(a13)
            tm5, tm9, tm13 = _count_by_load(fold["test_memory"])
            tc5, tc9, tc13 = _count_by_load(fold["test_control"])
            Trm5, Trm9, Trm13 = _count_by_load(fold["train_memory"])
            Trc5, Trc9, Trc13 = _count_by_load(fold["train_control"])

            print(f"  Fold {f}:")
            print(f"    TEST  mem: total={len(fold['test_memory'])}  by load: 05={tm5}, 09={tm9}, 13={tm13}")
            print(f"          ctrl: total={len(fold['test_control'])} by load: 05={tc5}, 09={tc9}, 13={tc13}")
            print(f"    TRAIN mem: total={len(fold['train_memory'])}  by load: 05={Trm5}, 09={Trm9}, 13={Trm13}")
            print(f"          ctrl: total={len(fold['train_control'])} by load: 05={Trc5}, 09={Trc9}, 13={Trc13}")

            if show_examples:
                tm_ex = ", ".join(_trial_tag(t) for t in fold["test_memory"][:show_examples])
                tc_ex = ", ".join(_trial_tag(t) for t in fold["test_control"][:show_examples])
                print(f"    examples TEST mem:  [{tm_ex}]")
                print(f"    examples TEST ctrl: [{tc_ex}]")

    # --- integrity checks (no leakage, full coverage, disjoint tests) ---
    if verbose:
        print("\n[Integrity checks]")
        # identity-based sets using id()
        all_mem_trials  = set(id(t) for t in buckets[("memory",5)] + buckets[("memory",9)] + buckets[("memory",13)])
        all_ctrl_trials = set(id(t) for t in buckets[("control",5)] + buckets[("control",9)] + buckets[("control",13)])

        # test coverage across folds
        mem_test_union  = set()
        ctrl_test_union = set()
        ok_disjoint = True
        seen_mem = set()
        seen_ctrl = set()

        for f, fold in enumerate(folds, 1):
            mem_ids  = [id(t) for t in fold["test_memory"]]
            ctrl_ids = [id(t) for t in fold["test_control"]]

            # disjointness of test sets across folds
            if mem_test_union.intersection(mem_ids) or ctrl_test_union.intersection(ctrl_ids):
                ok_disjoint = False
            mem_test_union.update(mem_ids)
            ctrl_test_union.update(ctrl_ids)

            # leakage: intersection between a fold's train and test
            leak_mem  = set(id(t) for t in fold["train_memory"]).intersection(mem_ids)
            leak_ctrl = set(id(t) for t in fold["train_control"]).intersection(ctrl_ids)
            print(f"  Fold {f} leakage mem={len(leak_mem)}  ctrl={len(leak_ctrl)}")

        print(f"  Test sets disjoint across folds? {'YES' if ok_disjoint else 'NO'}")
        print(f"  Memory test coverage {len(mem_test_union)}/{len(all_mem_trials)} "
              f"({len(mem_test_union)/max(1,len(all_mem_trials))*100:.1f}%)")
        print(f"  Control test coverage {len(ctrl_test_union)}/{len(all_ctrl_trials)} "
              f"({len(ctrl_test_union)/max(1,len(all_ctrl_trials))*100:.1f}%)")

    return folds


In [ ]:
import random
from math import atanh, sqrt
from scipy.stats import norm, ttest_rel, wilcoxon

import matplotlib.pyplot as plt

K = 3
all_rows = []  # collect all trial-level results here
# best_shift_by_sub = {}  # best shift per subject
rows_trials = []
rows_subject = []
rows_folds   = []   # per-fold diagnostics  
bad_subjects_mem = []
bad_subjects_ctrl = []

for subj in SUBJECTS:
    if subj == 57:
        continue
    print(f"Processing subject {subj:02d}...")
    trials = load_all_trials(subj)                         # list of (eeg, pupil)
    trials_memory, trials_control = split_trials_by_condition(trials)

    random.shuffle(trials_memory)
    random.shuffle(trials_control)

    # --- build stratified folds (mem/ctrl × load) ---
    folds = stratified_kfold_trials(trials_memory, trials_control, k=K, show_examples=0)
    if not folds:
        print(f"Subject {subj:02d} skipped – not enough trials for {K}-fold stratification in all buckets.")
        continue

    # Use your precomputed best lag
    if subj not in best_shift_by_sub:
        print(f"Subject {subj:02d} missing best_shift_by_sub – skipped")
        continue
    best_shift = int(best_shift_by_sub[subj])

    win = slice(WIN_OFFSET1, -WIN_OFFSET2)

    fold_rs_mem  = []
    fold_rs_ctrl = []

    for fold_idx, fold in enumerate(folds, start=1):
        print(f"Fold {fold_idx}: mem_test={len(fold['test_memory'])}, ctrl_test={len(fold['test_control'])}, mem_train={len(fold['train_memory'])}, ctrl_train={len(fold['train_control'])}")

        train_trials = fold["train_memory"] + fold["train_control"]
        rng.shuffle(train_trials)  # keep training order random but reproducible

        # concat
        X_train, Y_train   = concat_trials(train_trials,            shift=best_shift, win=win)
        X_mem_test, Y_mem  = concat_trials(fold["test_memory"],     shift=best_shift, win=win)
        X_ctl_test, Y_ctl  = concat_trials(fold["test_control"],    shift=best_shift, win=win)

        # fit CCA once on the training set
        cca = CCA(n_components=1, max_iter=1000, scale=False, tol=1e-6)
        cca.fit(X_train, Y_train)

        # evaluate MEMORY
        u_m, v_m = cca.transform(X_mem_test, Y_mem)
        r_mem, p_mem = pearsonr(u_m[:, 0], v_m[:, 0])

        # evaluate CONTROL
        u_c, v_c = cca.transform(X_ctl_test, Y_ctl)
        r_ctrl, p_ctrl = pearsonr(u_c[:, 0], v_c[:, 0])

        n_mem  = u_m.shape[0]
        n_ctrl = u_c.shape[0]
        z_mem  = np.arctanh(r_mem)
        z_ctrl = np.arctanh(r_ctrl)
        se     = np.sqrt(1/(n_mem - 3) + 1/(n_ctrl - 3))
        z_stat = (z_mem - z_ctrl) / se
        p_one_sided = 1 - norm.cdf(z_stat)  # H1: r_mem > r_ctrl

        # rows per condition
        """
        rows_subject.append({
            "subject": subj, "fold": fold_idx, "condition": "memory",
            "lag_ms": best_shift * 10, "r": float(r_mem), "p-value": float(p_mem)
        })
        rows_subject.append({
            "subject": subj, "fold": fold_idx, "condition": "control",
            "lag_ms": best_shift * 10, "r": float(r_ctrl), "p-value": float(p_ctrl)
        })

        # per-fold per-subject test
        rows_subject.append({
            "subject": subj, "fold": fold_idx, "condition": "per-subject test",
            "lag_ms": best_shift * 10, "r_mem": float(r_mem), "r_ctrl": float(r_ctrl),
            "n_mem": int(n_mem), "n_ctrl": int(n_ctrl),
            "z_stat": float(z_stat), "p_one_sided": float(p_one_sided),
        })
    """
        fold_rs_mem.append(r_mem)
        fold_rs_ctrl.append(r_ctrl)

        rows_folds.append({
            "subject": subj, "fold": fold,
            "lag_ms": best_shift * 10,
            "r_memory_holdout": float(r_mem),
            "r_control_all":    float(r_ctrl)
        })
        # optional: flag “bad” per fold
        if r_ctrl < 0.05: bad_subjects_ctrl.append((subj, fold_idx))
        if r_mem  < 0.05: bad_subjects_mem.append((subj, fold_idx))

        # save weights per fold (keeps your existing helper)
        save_cca_weights(
            cca,
            subject_tag=f"sub-{subj:03d}",
            lag_ms=best_shift * 10,
            condition=f"mem_fold{fold_idx}",   # add fold tag
            eeg_ch_names=FRONTAL_MIDLINE,
            out_dir=Path("weights_k3_StratLoadHILBERT")
        )
    
    mem_mean  = float(np.mean(fold_rs_mem))
    ctrl_mean = float(np.mean(fold_rs_ctrl))

    rows_subject.append({"subject": subj, "condition": "memory",  "r": mem_mean})
    rows_subject.append({"subject": subj, "condition": "control", "r": ctrl_mean})


df_subj = pd.DataFrame(rows_subject)
df_wide = df_subj.pivot(index="subject", columns="condition", values="r")

dfree = len(df_wide) - 1
t2, p2 = stats.ttest_rel(df_wide["memory"], df_wide["control"])
p_one = p2/2 if t2 > 0 else 1 - p2/2
print(f"Paired t-test: t={t2:.3f}, p_two={p2:.4f}, p_one(mem>ctl)={p_one:.4f}, df={dfree}")



In [ ]:
pd.DataFrame(rows_subject).to_csv("subject_level_cca_fixedlag_k3_stratCVHILBERT.csv", index=False)
print("Finished – saved subject_level_cca_fixedlag_k3_stratCVHILBERT.csv")


In [ ]:
df_subj = pd.read_csv("subject_level_cca_fixedlag_k3_stratCVHILBERT.csv")

# Calculate and print mean correlations for each condition
mean_memory = df_subj[df_subj['condition'] == 'memory']['r'].mean()
mean_control = df_subj[df_subj['condition'] == 'control']['r'].mean()
print(f"Mean correlation for memory condition: {mean_memory:.4f}")
print(f"Mean correlation for control condition: {mean_control:.4f}")
print()

# Pivot so we have columns for memory and control per subject
df_wide = df_subj.pivot(index="subject", columns="condition", values="r")
print(df_wide.head(20))

# --- 95% CI for the paired mean difference (mem − ctl) ---
paired = df_wide.dropna(subset=["memory", "control"])
diff   = paired["memory"] - paired["control"]

n      = diff.size
dfree  = n - 1
mean_d = diff.mean()
se_d   = diff.std(ddof=1) / np.sqrt(n)

alpha  = 0.05
tcrit  = stats.t.ppf(1 - alpha/2, dfree)   # two-sided 95%
ci_low, ci_high = mean_d - tcrit*se_d, mean_d + tcrit*se_d

# (optional) re-run t-test on the same paired subset
t2, p2 = stats.ttest_rel(paired["memory"], paired["control"])
p_one  = p2/2 if t2 > 0 else 1 - p2/2

print(f"Paired t-test: t={t2:.3f}, p_two={p2:.4f}, "
      f"p_one(mem>ctl)={p_one:.4f}, df={dfree}, "
      f"meanΔ={mean_d:.3f}, 95% CI=[{ci_low:.3f}, {ci_high:.3f}]")